# NLP keyword features — robust mapping

A more granular sibling of `04_nlp_keywords.ipynb`. Same length-normalised
keyword-density approach, but ~49 fine-grained concept axes (individual fruits,
oak markers, structure, producer/style cues) instead of 8 broad ones.

Each concept is one regex of word-boundaried synonyms; per review we count hits
and normalise by length. Saved to a **separate** artifact
`features_keywords_robust.parquet` (does *not* overwrite `features_keywords`),
so it can be wired into the model notebooks as an alternative `kw` block.

In [2]:
import re
import numpy as np
import pandas as pd
import itables
from itables import show

itables.options.columnDefs = [{"className": "dt-left", "targets": "_all"}]

SILVER_PATH   = r"..\..\.data\wine_reviews_silver.parquet"
KEYWORDS_PATH = r"..\..\features\features_keywords_robust.parquet"

df = pd.read_parquet(SILVER_PATH)
print(f"Silver shape: {df.shape}")
print(f"reviews missing: {df['review'].isna().sum():,}  (scored as empty string)")

Silver shape: (135192, 24)
reviews missing: 17  (scored as empty string)


## Keyword dictionaries

Keys are the concept axes; values are the matched synonyms/variants. A few notes:
`red` / `black` mean the red- and black-fruit *families* (multiword phrases, to
avoid matching "red wine"); `bell` is bell-pepper; `body`/`light`/`heavy` overlap
by design (different facets of weight). Edit freely — everything below is driven
by this dict.

In [ ]:
FEATURE_KEYWORDS = {
    # --- taste / structure ---
    "dry":        ["dry", "bone dry", "bone-dry", "dryness"],
    "acidic":     ["acid", "acidity", "acidic"],
    "tart":       ["tart", "tangy", "sour", "zesty", "zest"],
    "sweet":      ["sweet", "sweetness", "sugary", "sugar", "honeyed"],
    "caramel":    ["caramel", "butterscotch", "toffee"],
    "alcohol":    ["alcohol", "alcoholic", "boozy", "hot", "warming"],
    "strong":     ["strong", "powerful", "robust", "intense", "muscular"],
    "balanced":    ["balanced", "harmony", "harmonious"],
    # --- citrus / orchard ---
    "citrus":     ["citrus", "lemon", "lime", "grapefruit", "orange", "tangerine"],
    "apple":      ["apple", "apples"],
    "pear":       ["pear", "pears"],
    # --- red / black fruit ---
    "strawberry": ["strawberry", "strawberries"],
    "raspberry":  ["raspberry", "raspberries"],
    "cherry":     ["cherry", "cherries"],
    "red":        ["red fruit", "red fruits", "red berry", "red berries", "redcurrant", "red currant"],
    "black fruits":      ["black fruit", "black fruits", "black currant", "blackcurrant", "cassis", "blackberry", "blackberries"],
    # --- tropical / stone ---
    "tropical":   ["tropical", "mango", "guava", "passion fruit", "passionfruit", "papaya"],
    "banana":     ["banana", "bananas"],
    "pineapple":  ["pineapple", "pineapples"],
    "lichi":      ["lichi", "lychee", "litchi"],
    "stone":      ["stone fruit", "stone fruits", "stonefruit", "apricot", "nectarine"],
    "peach":      ["peach", "peaches"],
    # --- earth / soil ---
    "soil":       ["soil", "potting soil", "topsoil"],
    "mineral":    ["mineral", "minerality", "wet stone", "flint", "chalk"],
    # --- oak / barrel ---
    "oak":        ["oak", "oaky", "oaked", "barrel", "barrique"],
    "coconut":    ["coconut"],
    "vanilla":    ["vanilla"],
    "smoke":      ["smoke", "smoky", "smoked", "smokiness"],
    # --- tannin / body ---
    "tannic":     ["tannic", "tannin", "tannins", "grippy", "grip"],
    "light":      ["light", "light-bodied", "delicate"],
    "heavy":      ["heavy", "full-bodied", "weighty", "dense"],
    "body":       ["body", "bodied", "mouthfeel", "texture", "medium-bodied"],
    # --- floral / herbal / green ---
    "flowers":    ["flower", "flowers", "blossom", "rose", "violet", "jasmine"],
    "floral":     ["floral"],
    "grass":      ["grass", "grassy"],
    "herbs":      ["herb", "herbs", "herbal", "thyme", "sage", "rosemary", "mint"],
    "spicy":      ["spicy", "spice", "spices"],
    "vegetables": ["vegetal", "vegetable", "vegetables"],
    "pepper":     ["pepper", "peppery", "peppercorn"],
    "bell":       ["bell pepper", "bell peppers", "capsicum"],
    "earth":      ["earth", "earthy", "mineral", "minerality", "wet stone"],
    "leather":    ["leather", "leathery"],
    "tea":        ["tea", "black tea", "green tea", "tea leaf"],
    # --- producer / style cues ---
    "family":     ["family", "family-owned", "family-run"],
    "artisan":    ["artisan", "artisanal", "handcrafted", "hand-crafted"],
    "biodynamic": ["bio", "biodynamic", "biodynamics"],
    "ecologic":   ["ecologic", "ecology", "ecological", "sustainable", "sustainably"],
    "natural":    ["natural", "organic", "naturally"],
    "summer":     ["summer", "summery"],
    "serious":    ["serious", "seriously"],
    "elegant":    ["elegant", "elegance"],
    "refreshing": ["refreshing", "refreshment"],
}

print(f"{len(FEATURE_KEYWORDS)} concepts, "
      f"{sum(len(v) for v in FEATURE_KEYWORDS.values())} keywords total")

53 concepts, 183 keywords total


## Compile patterns and score

One case-insensitive, word-boundaried alternation regex per concept (longest
alternatives first so multiword phrases win). `kw_<c>_count` = raw hits,
`kw_<c>` = hits per 100 words.

In [ ]:
def build_patterns(keyword_dict):
    patterns = {}
    for feature, words in keyword_dict.items():
        alt = "|".join(re.escape(w) for w in sorted(words, key=len, reverse=True))
        patterns[feature] = re.compile(rf"\b(?:{alt})\b", re.IGNORECASE)
    return patterns


WORD_RE = re.compile(r"\b\w+\b")
PATTERNS = build_patterns(FEATURE_KEYWORDS)


def score_review(text):
    text = text or ""
    n_words = max(len(WORD_RE.findall(text)), 1)
    out = {}
    for feature, pat in PATTERNS.items():
        hits = len(pat.findall(text))
        out[f"kw_{feature}_count"] = hits
        out[f"kw_{feature}"] = round(100 * hits / n_words, 3)
    return out


reviews = df["review"].fillna("").astype(str)
kw_df = pd.DataFrame.from_records([score_review(t) for t in reviews], index=df.index)

count_cols   = [c for c in kw_df.columns if c.endswith("_count")]
density_cols = [c for c in kw_df.columns if not c.endswith("_count")]
print(f"Scored {len(kw_df):,} reviews -> {len(count_cols)} count + {len(density_cols)} density cols")
kw_df[density_cols].head()
# kw_df[count_cols].head()

Scored 135,192 reviews -> 53 count + 53 density cols


,kw_dry_count,kw_acidic_count,kw_tart_count,kw_sweet_count,kw_caramel_count,kw_alcohol_count,kw_strong_count,kw_balanced_count,kw_citrus_count,kw_apple_count,...,kw_tea_count,kw_family_count,kw_artisan_count,kw_biodynamic_count,kw_ecologic_count,kw_natural_count,kw_summer_count,kw_serious_count,kw_elegant_count,kw_refreshing_count
0,0,0,0,1,0,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


## Coverage and distribution

Share of reviews mentioning each concept at least once (sorted).

In [5]:
coverage = (kw_df[count_cols] > 0).mean().rename("pct_reviews_with_hit").mul(100).round(1)
avg_hits = kw_df[count_cols].mean().rename("avg_count").round(3)
summary = pd.concat([coverage, avg_hits], axis=1)
summary.index = summary.index.str.replace("kw_", "").str.replace("_count", "")
summary = summary.sort_values("pct_reviews_with_hit", ascending=False)
show(summary.reset_index().rename(columns={"index": "concept"}))

Loading ITables v2.7.3 from the internet... (need help?)


## Spot-check

Sample reviews against their top concepts.

In [6]:
sample_idx = df.sample(8, random_state=7).index
sub = kw_df.loc[sample_idx, count_cols]
for i in sample_idx:
    hits = sub.loc[i]
    top = hits[hits > 0].sort_values(ascending=False)
    tags = ", ".join(f"{c.replace('kw_', '').replace('_count', '')}({int(v)})" for c, v in top.items()) or "(none)"
    print(f"- {str(df.loc[i, 'review'])[:160]}")
    print(f"    -> {tags}\n")

- Lane Tanner, the first independent female winemaker in Santa Barbara County history, is applying her Pinot Noir acumen to Grenache, with great results. This bri
    -> raspberry(1), flowers(1), herbs(1), pepper(1), biodynamic(1)

- This wine is from the Manilla vineyard, which sits on volcanic soils and is situated in a natural clos, or protected area that results in more humidity and clim
    -> acidic(1), tart(1), citrus(1), stone(1), mineral(1), body(1), floral(1), natural(1)

- The aromatic combination of sweet petunias, blackberry jam and hoisin sauce is a winning one. Muscular tannins support bowls of blackberry and cherry flavors, a
    -> blackberry(2), caramel(1), sweet(1), strong(1), cherry(1), tannic(1), tea(1)

- Aromas of pepper, wet slate tart blackberries and wild fennel swirl together to create a refreshing, invigorating nose. The palate layers more blackberries over
    -> blackberry(2), acidic(1), tart(1), citrus(1), tannic(1), pepper(1), tea(1), refreshing(1)

- Th

## Signal check vs. price & rating

Correlation of each concept density with the two targets — which descriptors
track price vs the score.

In [7]:
signal = (
    pd.concat([kw_df[density_cols], df[["retail", "rating"]]], axis=1)
    .corr()[["retail", "rating"]]
    .drop(index=["retail", "rating"])
    .round(3)
)
signal.index = signal.index.str.replace("kw_", "")
show(signal.sort_values("rating", ascending=False).reset_index().rename(columns={"index": "concept"}))

Loading ITables v2.7.3 from the internet... (need help?)


## Save

`wine_id` + all `kw_*` columns → `features_keywords_robust.parquet`. To try it in
a model notebook, point `KEYWORDS_PATH` there at this file (or load both and pick
the `kw_` block).

In [8]:
out = pd.concat([df[["wine_id"]], kw_df], axis=1)
out.to_parquet(KEYWORDS_PATH, index=False)
print(f"Saved {out.shape[0]:,} rows x {out.shape[1]} cols -> {KEYWORDS_PATH}")

Saved 135,192 rows x 107 cols -> ..\..\features\features_keywords_robust.parquet
